In [47]:
!pip install pandas numpy scikit-learn matplotlib pyarrow seaborn

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [48]:
import pandas as pd
import os

print("Files in current directory:")
print(os.listdir())


Files in current directory:
['ESCP_DETAILED_RECEIPT_2026_V2.parquet', '05_segment_engagement.png', '04_segment_scatter.png', 'shopper_segments.csv', 'ESCP_STORE_2026_V2.parquet', 'Segmentation_Model.ipynb', 'ESCP_CONSUMER_2026_V2.parquet', 'ESCP_EVENTS_2026_V2.parquet', '02_segment_spend.png', '03_segment_frequency.png', '01_segment_size.png', 'ESCP_OVERALL_RECEIPT_2026_V2.parquet']


In [49]:
import os

print("You are here:")
print(os.getcwd())

You are here:
/Users/mya/Downloads/CATALINA-hackathon 


In [50]:
import pandas as pd
import os

os.chdir(os.path.expanduser('~/Downloads/CATALINA-hackathon '))

overall_receipt = pd.read_parquet("ESCP_OVERALL_RECEIPT_2026_V2.parquet")
detailed_receipt = pd.read_parquet("ESCP_DETAILED_RECEIPT_2026_V2.parquet")
consumer = pd.read_parquet("ESCP_CONSUMER_2026_V2.parquet")
store = pd.read_parquet("ESCP_STORE_2026_V2.parquet")
events = pd.read_parquet("ESCP_EVENTS_2026_V2.parquet")

print("✅ All files loaded successfully!")
print(f"Overall Receipt: {len(overall_receipt):,} rows")
print(f"Detailed Receipt: {len(detailed_receipt):,} rows")
print(f"Consumer: {len(consumer):,} rows")
print(f"Store: {len(store):,} rows")
print(f"Events: {len(events):,} rows")


✅ All files loaded successfully!
Overall Receipt: 2,642,418 rows
Detailed Receipt: 23,864,344 rows
Consumer: 366,696 rows
Store: 44 rows
Events: 759,666 rows


In [51]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [52]:
print("=" * 80)
print("STEP 1: LOADING DATA")
print("=" * 80)

overall_receipt = pd.read_parquet("ESCP_OVERALL_RECEIPT_2026_V2.parquet")
detailed_receipt = pd.read_parquet("ESCP_DETAILED_RECEIPT_2026_V2.parquet")
consumer = pd.read_parquet("ESCP_CONSUMER_2026_V2.parquet")
store = pd.read_parquet("ESCP_STORE_2026_V2.parquet")
events = pd.read_parquet("ESCP_EVENTS_2026_V2.parquet")

print(f"Loaded {len(overall_receipt):,} overall transactions")
print(f"Loaded {len(consumer):,} unique shoppers")
print(f"Loaded {len(events):,} offer events")


STEP 1: LOADING DATA
Loaded 2,642,418 overall transactions
Loaded 366,696 unique shoppers
Loaded 759,666 offer events


In [53]:
# STEP 2: CREATE SHOPPER-LEVEL BEHAVIOR METRICS
print("\n" + "=" * 80)
print("STEP 2: CALCULATING SHOPPER BEHAVIOR METRICS")
print("=" * 80)

shopper_behavior = overall_receipt.groupby('loyalty_card_key').agg({
    'tot_ord_amt': ['sum', 'mean', 'count'],
    'tot_trade_item_purch_qty': 'mean',
    'date': ['min', 'max'],
    'channel': lambda x: (x == 'ECOMMERCE').sum() / len(x)
}).reset_index()

shopper_behavior.columns = ['shopper_id', 'total_spend', 'avg_transaction_value', 
                             'num_transactions', 'avg_items_per_basket', 
                             'first_purchase_date', 'last_purchase_date', 'pct_ecommerce']

shopper_behavior['first_purchase_date'] = pd.to_datetime(shopper_behavior['first_purchase_date'])
shopper_behavior['last_purchase_date'] = pd.to_datetime(shopper_behavior['last_purchase_date'])

shopper_behavior['tenure_days'] = (shopper_behavior['last_purchase_date'] - shopper_behavior['first_purchase_date']).dt.days
shopper_behavior['tenure_days'] = shopper_behavior['tenure_days'].replace(0, 1)

shopper_behavior['frequency_per_month'] = (shopper_behavior['num_transactions'] / (shopper_behavior['tenure_days'] / 30)).fillna(0)

print(f"✅ Calculated metrics for {len(shopper_behavior):,} shoppers")


STEP 2: CALCULATING SHOPPER BEHAVIOR METRICS
✅ Calculated metrics for 366,696 shoppers


In [54]:
# STEP 3: ADD STORE LOYALTY
print("\n" + "=" * 80)
print("STEP 3: CALCULATING STORE LOYALTY")
print("=" * 80)

store_visits = overall_receipt.groupby(['loyalty_card_key', 'store_key']).size().reset_index(name='visits')

most_visited_store = store_visits.sort_values(['loyalty_card_key', 'visits'], ascending=[True, False])
most_visited_store = most_visited_store.drop_duplicates('loyalty_card_key')[['loyalty_card_key', 'store_key']]
most_visited_store.columns = ['shopper_id', 'primary_store']

store_loyalty_pct = store_visits.groupby('loyalty_card_key').apply(
    lambda x: x.sort_values('visits', ascending=False).iloc[0]['visits'] / x['visits'].sum()
).reset_index()
store_loyalty_pct.columns = ['shopper_id', 'primary_store_loyalty']

shopper_behavior = shopper_behavior.merge(most_visited_store, on='shopper_id', how='left')
shopper_behavior = shopper_behavior.merge(store_loyalty_pct, on='shopper_id', how='left')

print(f"Added store loyalty metrics")




STEP 3: CALCULATING STORE LOYALTY


/var/folders/yr/_yfgn1414zsgmlkf3n05wr3c0000gn/T/ipykernel_5627/1216064004.py:12: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  store_loyalty_pct = store_visits.groupby('loyalty_card_key').apply(


Added store loyalty metrics


In [55]:
# STEP 4: ADD OFFER ENGAGEMENT
print("\n" + "=" * 80)
print("STEP 4: CALCULATING OFFER ENGAGEMENT")
print("=" * 80)

print(f"Unique status labels: {events['status_label'].unique()}")

activations = events[events['status_label'] == 'ACTIVATION'].groupby('loyalty_card_key').size().reset_index(name='offers_activated')
validations = events[events['status_label'] == 'VALIDATION'].groupby('loyalty_card_key').size().reset_index(name='offers_redeemed')

offer_engagement = activations.merge(validations, on='loyalty_card_key', how='left')
offer_engagement.columns = ['shopper_id', 'offers_activated', 'offers_redeemed']
offer_engagement['redemption_rate'] = (offer_engagement['offers_redeemed'] / offer_engagement['offers_activated']).fillna(0)

shopper_behavior = shopper_behavior.merge(offer_engagement, on='shopper_id', how='left')
shopper_behavior['offers_activated'] = shopper_behavior['offers_activated'].fillna(0)
shopper_behavior['offers_redeemed'] = shopper_behavior['offers_redeemed'].fillna(0)
shopper_behavior['redemption_rate'] = shopper_behavior['redemption_rate'].fillna(0)

print(f"Added offer engagement metrics")


STEP 4: CALCULATING OFFER ENGAGEMENT
Unique status labels: ['ACTIVATION' 'VALIDATION']
Added offer engagement metrics


In [68]:
# Promotion responsiveness
shopper_behavior['redemption_rate']
print(f"Redemption rate - Min: {shopper_behavior['redemption_rate'].min():.2f}")
print(f"Redemption rate - Max: {shopper_behavior['redemption_rate'].max():.2f}")
print(f"Redemption rate - Mean: {shopper_behavior['redemption_rate'].mean():.2f}")


Redemption rate - Min: 0.00
Redemption rate - Max: 72.00
Redemption rate - Mean: 0.05


In [69]:
# Promotion transactions (offer_flag column may not exist in all data versions)
if "offer_flag" in detailed_receipt.columns:
    promo_transactions = detailed_receipt[detailed_receipt["offer_flag"] == 1]
    promo_counts = promo_transactions.groupby("loyalty_card_key")["transaction_id"].nunique()
    shopper_behavior["promo_transactions"] = shopper_behavior["shopper_id"].map(promo_counts).fillna(0)
else:
    shopper_behavior["promo_transactions"] = 0
    print("Note: offer_flag not found in detailed_receipt — promo_transactions set to 0")


KeyError: 'offer_flag'

In [56]:
# STEP 5: CREATE BEHAVIORAL SEGMENTS
print("\n" + "=" * 80)
print("STEP 5: CREATING BEHAVIORAL SEGMENTS")
print("=" * 80)

spend_q75 = shopper_behavior['total_spend'].quantile(0.75)
spend_q50 = shopper_behavior['total_spend'].quantile(0.50)
freq_q75 = shopper_behavior['frequency_per_month'].quantile(0.75)
freq_q50 = shopper_behavior['frequency_per_month'].quantile(0.50)

def segment_shopper(row):
    if row['total_spend'] > spend_q75 and row['frequency_per_month'] > freq_q50:
        return 'Premium Loyal'
    elif row['total_spend'] > spend_q50 and row['frequency_per_month'] > freq_q50:
        return 'Regular Engaged'
    else:
        return 'Occasional'

shopper_behavior['segment'] = shopper_behavior.apply(segment_shopper, axis=1)

segment_summary = shopper_behavior.groupby('segment').agg({
    'shopper_id': 'count',
    'total_spend': ['mean', 'median'],
    'frequency_per_month': 'mean',
    'num_transactions': 'mean',
    'redemption_rate': 'mean',
    'primary_store_loyalty': 'mean'
}).round(2)

print("\nSEGMENT BREAKDOWN:")
print(segment_summary)


STEP 5: CREATING BEHAVIORAL SEGMENTS

SEGMENT BREAKDOWN:
                shopper_id total_spend          frequency_per_month  \
                     count        mean   median                mean   
segment                                                               
Occasional          306456      268.37   102.63               14.00   
Premium Loyal        37483     1405.63  1202.33               10.80   
Regular Engaged      22757      267.86   243.37               22.28   

                num_transactions redemption_rate primary_store_loyalty  
                            mean            mean                  mean  
segment                                                                 
Occasional                  4.54            0.05                   1.0  
Premium Loyal              28.52            0.14                   1.0  
Regular Engaged             7.94            0.04                   1.0  


In [57]:
# STEP 6: DETAILED PERSONAS
print("\n" + "=" * 80)
print("STEP 6: DETAILED PERSONA PROFILES")
print("=" * 80)

for segment_name in sorted(shopper_behavior['segment'].unique()):
    segment_data = shopper_behavior[shopper_behavior['segment'] == segment_name]
    
    print(f"\n{'='*70}")
    print(f"SEGMENT: {segment_name.upper()}")
    print(f"{'='*70}")
    
    print(f"Size: {len(segment_data):,} shoppers ({len(segment_data)/len(shopper_behavior)*100:.1f}% of base)")
    print(f"\n SPENDING:")
    print(f"  Annual spend: €{segment_data['total_spend'].mean():.2f} (median: €{segment_data['total_spend'].median():.2f})")
    print(f"  Avg transaction: €{segment_data['avg_transaction_value'].mean():.2f}")
    
    print(f"\n BEHAVIOR:")
    print(f"  Frequency: {segment_data['frequency_per_month'].mean():.2f} times/month")
    print(f"  Items per basket: {segment_data['avg_items_per_basket'].mean():.1f}")
    print(f"  E-commerce: {segment_data['pct_ecommerce'].mean():.1%}")
    
    print(f"\n LOYALTY:")
    print(f"  Store loyalty: {segment_data['primary_store_loyalty'].mean():.1%}")
    print(f"  Tenure: {segment_data['tenure_days'].mean()/365:.1f} years")
    
    print(f"\n OFFERS:")
    engaged = (segment_data['offers_activated'] > 0).sum()
    print(f"  % engaged: {engaged / len(segment_data) * 100:.1f}%")
    print(f"  Avg activated: {segment_data['offers_activated'].mean():.1f}")
    if (segment_data['offers_redeemed'] > 0).sum() > 0:
        print(f"  Redemption: {segment_data[segment_data['offers_redeemed'] > 0]['redemption_rate'].mean():.1%}")



STEP 6: DETAILED PERSONA PROFILES

SEGMENT: OCCASIONAL
Size: 306,456 shoppers (83.6% of base)

 SPENDING:
  Annual spend: €268.37 (median: €102.63)
  Avg transaction: €51.86

 BEHAVIOR:
  Frequency: 14.00 times/month
  Items per basket: 14.8
  E-commerce: 1.4%

 LOYALTY:
  Store loyalty: 99.9%
  Tenure: 0.1 years

 OFFERS:
  % engaged: 30.3%
  Avg activated: 1.7
  Redemption: 68.2%

SEGMENT: PREMIUM LOYAL
Size: 37,483 shoppers (10.2% of base)

 SPENDING:
  Annual spend: €1405.63 (median: €1202.33)
  Avg transaction: €61.36

 BEHAVIOR:
  Frequency: 10.80 times/month
  Items per basket: 16.9
  E-commerce: 1.2%

 LOYALTY:
  Store loyalty: 99.9%
  Tenure: 0.2 years

 OFFERS:
  % engaged: 35.7%
  Avg activated: 2.9
  Redemption: 80.7%

SEGMENT: REGULAR ENGAGED
Size: 22,757 shoppers (6.2% of base)

 SPENDING:
  Annual spend: €267.86 (median: €243.37)
  Avg transaction: €114.48

 BEHAVIOR:
  Frequency: 22.28 times/month
  Items per basket: 28.6
  E-commerce: 2.0%

 LOYALTY:
  Store loyalty: 

In [58]:
# STEP 7: SAVE RESULTS
print("\n" + "=" * 80)
print("STEP 7: SAVING RESULTS")
print("=" * 80)

shopper_behavior.to_csv('shopper_segments.csv', index=False)
print("✅ Saved: shopper_segments.csv")

print("\n" + "=" * 80)
print("✅ SEGMENTATION COMPLETE!")
print("=" * 80)


STEP 7: SAVING RESULTS
✅ Saved: shopper_segments.csv

✅ SEGMENTATION COMPLETE!


In [59]:
print("DIAGNOSTIC CHECK:")
print("\n1. Frequency calculation:")
print(f"   Min frequency: {shopper_behavior['frequency_per_month'].min():.2f}")
print(f"   Max frequency: {shopper_behavior['frequency_per_month'].max():.2f}")
print(f"   Mean: {shopper_behavior['frequency_per_month'].mean():.2f}")

print("\n2. Tenure check:")
print(f"   Min tenure_days: {shopper_behavior['tenure_days'].min()}")
print(f"   Max tenure_days: {shopper_behavior['tenure_days'].max()}")
print(f"   Mean tenure_days: {shopper_behavior['tenure_days'].mean():.0f}")

print("\n3. Store loyalty check:")
print(f"   Min: {shopper_behavior['primary_store_loyalty'].min():.2f}")
print(f"   Max: {shopper_behavior['primary_store_loyalty'].max():.2f}")
print(f"   Mean: {shopper_behavior['primary_store_loyalty'].mean():.2f}")

print("\n4. Sample data:")
print(shopper_behavior[['total_spend', 'num_transactions', 'tenure_days', 'frequency_per_month', 'primary_store_loyalty', 'segment']].head(20))

DIAGNOSTIC CHECK:

1. Frequency calculation:
   Min frequency: 0.66
   Max frequency: 210.00
   Mean: 14.19

2. Tenure check:
   Min tenure_days: 1
   Max tenure_days: 91
   Mean tenure_days: 41

3. Store loyalty check:
   Min: 0.50
   Max: 1.00
   Mean: 1.00

4. Sample data:
    total_spend  num_transactions  tenure_days  frequency_per_month  \
0        101.92                 4           46             2.608696   
1        176.17                 5           63             2.380952   
2          8.70                 1            1            30.000000   
3        185.16                 3           50             1.800000   
4         45.88                 1            1            30.000000   
5         23.21                 1            1            30.000000   
6        147.52                 8           85             2.823529   
7         60.34                 1            1            30.000000   
8        494.09                13           59             6.610169   
9         89.

In [60]:
shopper_behavior = pd.read_csv('shopper_segments.csv', low_memory=False)

print("=" * 80)
print("STEP 8: CREATING VISUALIZATIONS")
print("=" * 80)

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
colors = {'Premium Loyal': '#2ecc71', 'Regular Engaged': '#3498db', 'Occasional': '#e74c3c'}
segment_order = ['Premium Loyal', 'Regular Engaged', 'Occasional']

# Visualization 1: Segment Size
fig, ax = plt.subplots(figsize=(12, 6))
segment_counts = shopper_behavior['segment'].value_counts()
segment_counts = segment_counts.reindex(segment_order)
bars = ax.bar(range(len(segment_counts)), segment_counts.values, 
              color=[colors[seg] for seg in segment_counts.index], 
              edgecolor='black', linewidth=2, alpha=0.8)
ax.set_ylabel('Number of Shoppers', fontsize=12, fontweight='bold')
ax.set_xlabel('Segment', fontsize=12, fontweight='bold')
ax.set_title('Shopper Segmentation - Size Distribution', fontsize=14, fontweight='bold')
ax.set_xticks(range(len(segment_counts)))
ax.set_xticklabels(segment_counts.index, rotation=0)
for bar in bars:
    height = bar.get_height()
    pct = height / len(shopper_behavior) * 100
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{int(height):,}\n({pct:.1f}%)',
            ha='center', va='bottom', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('01_segment_size.png', dpi=300, bbox_inches='tight')
print("Saved: 01_segment_size.png")
plt.close()

STEP 8: CREATING VISUALIZATIONS
Saved: 01_segment_size.png


In [61]:
# Visualization 2: Spend by Segment
fig, ax = plt.subplots(figsize=(12, 6))
shopper_behavior_sorted = shopper_behavior.copy()
shopper_behavior_sorted['segment'] = pd.Categorical(shopper_behavior_sorted['segment'], 
                                                     categories=segment_order, ordered=True)
bp = ax.boxplot([shopper_behavior_sorted[shopper_behavior_sorted['segment'] == seg]['total_spend'].values 
                  for seg in segment_order],
                labels=segment_order,
                patch_artist=True,
                widths=0.6)
for patch, segment in zip(bp['boxes'], segment_order):
    patch.set_facecolor(colors[segment])
    patch.set_alpha(0.8)
ax.set_ylabel('Annual Spend (€)', fontsize=12, fontweight='bold')
ax.set_xlabel('Segment', fontsize=12, fontweight='bold')
ax.set_title('Shopper Segmentation - Annual Spend Distribution', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('02_segment_spend.png', dpi=300, bbox_inches='tight')
print("Saved: 02_segment_spend.png")
plt.close()


/var/folders/yr/_yfgn1414zsgmlkf3n05wr3c0000gn/T/ipykernel_5627/857894233.py:6: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot([shopper_behavior_sorted[shopper_behavior_sorted['segment'] == seg]['total_spend'].values


Saved: 02_segment_spend.png


In [62]:
# Visualization 2: Spend by Segment
fig, ax = plt.subplots(figsize=(12, 6))
shopper_behavior_sorted = shopper_behavior.copy()
shopper_behavior_sorted['segment'] = pd.Categorical(shopper_behavior_sorted['segment'], 
                                                     categories=segment_order, ordered=True)
bp = ax.boxplot([shopper_behavior_sorted[shopper_behavior_sorted['segment'] == seg]['total_spend'].values 
                  for seg in segment_order],
                labels=segment_order,
                patch_artist=True,
                widths=0.6)
for patch, segment in zip(bp['boxes'], segment_order):
    patch.set_facecolor(colors[segment])
    patch.set_alpha(0.8)
ax.set_ylabel('Annual Spend (€)', fontsize=12, fontweight='bold')
ax.set_xlabel('Segment', fontsize=12, fontweight='bold')
ax.set_title('Shopper Segmentation - Annual Spend Distribution', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('02_segment_spend.png', dpi=300, bbox_inches='tight')
print("Saved: 02_segment_spend.png")
plt.close()


/var/folders/yr/_yfgn1414zsgmlkf3n05wr3c0000gn/T/ipykernel_5627/857894233.py:6: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot([shopper_behavior_sorted[shopper_behavior_sorted['segment'] == seg]['total_spend'].values


Saved: 02_segment_spend.png


In [63]:
# Visualization 3: Frequency by Segment
fig, ax = plt.subplots(figsize=(12, 6))
bp = ax.boxplot([shopper_behavior_sorted[shopper_behavior_sorted['segment'] == seg]['frequency_per_month'].values 
                  for seg in segment_order],
                labels=segment_order,
                patch_artist=True,
                widths=0.6)
for patch, segment in zip(bp['boxes'], segment_order):
    patch.set_facecolor(colors[segment])
    patch.set_alpha(0.8)
ax.set_ylabel('Purchase Frequency (transactions/month)', fontsize=12, fontweight='bold')
ax.set_xlabel('Segment', fontsize=12, fontweight='bold')
ax.set_title('Shopper Segmentation - Purchase Frequency Distribution', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('03_segment_frequency.png', dpi=300, bbox_inches='tight')
print("Saved: 03_segment_frequency.png")
plt.close()

Saved: 03_segment_frequency.png


/var/folders/yr/_yfgn1414zsgmlkf3n05wr3c0000gn/T/ipykernel_5627/387220973.py:3: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot([shopper_behavior_sorted[shopper_behavior_sorted['segment'] == seg]['frequency_per_month'].values


In [64]:
# Visualization 4: Spend vs Frequency Scatter
fig, ax = plt.subplots(figsize=(12, 8))
for seg in segment_order:
    seg_data = shopper_behavior_sorted[shopper_behavior_sorted['segment'] == seg]
    ax.scatter(seg_data['frequency_per_month'], seg_data['total_spend'], 
              label=seg, alpha=0.6, s=50, color=colors[seg], edgecolors='black', linewidth=0.5)
ax.set_xlabel('Purchase Frequency (per month)', fontsize=12, fontweight='bold')
ax.set_ylabel('Annual Spend (€)', fontsize=12, fontweight='bold')
ax.set_title('Shopper Segmentation - Spend vs Frequency\n(Shows Natural Clustering)', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('04_segment_scatter.png', dpi=300, bbox_inches='tight')
print("Saved: 04_segment_scatter.png")
plt.close()

Saved: 04_segment_scatter.png


In [65]:
# Visualization 5: Offer Engagement
fig, ax = plt.subplots(figsize=(12, 6))
engagement_by_segment = []
for seg in segment_order:
    seg_data = shopper_behavior_sorted[shopper_behavior_sorted['segment'] == seg]
    pct_engaged = (seg_data['offers_activated'] > 0).sum() / len(seg_data) * 100
    engagement_by_segment.append(pct_engaged)

bars = ax.bar(range(len(segment_order)), engagement_by_segment,
              color=[colors[seg] for seg in segment_order],
              edgecolor='black', linewidth=2, alpha=0.8)
ax.set_ylabel('% of Shoppers Engaged with Offers', fontsize=12, fontweight='bold')
ax.set_xlabel('Segment', fontsize=12, fontweight='bold')
ax.set_title('Shopper Segmentation - Offer Engagement Rate', fontsize=14, fontweight='bold')
ax.set_xticks(range(len(segment_order)))
ax.set_xticklabels(segment_order, rotation=0)
ax.set_ylim(0, 50)
for bar, val in zip(bars, engagement_by_segment):
    ax.text(bar.get_x() + bar.get_width()/2., val,
            f'{val:.1f}%',
            ha='center', va='bottom', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('05_segment_engagement.png', dpi=300, bbox_inches='tight')
print("Saved: 05_segment_engagement.png")
plt.close()

Saved: 05_segment_engagement.png
